
# Reto proyecto 2 - Recomendador de películas con word embeddings

En este notebook se construye un sistema de recomendación de películas a partir de la similitud entre sinopsis usando Word2Vec y similitud del coseno.


In [1]:

from pathlib import Path
import re
import difflib

import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity


In [2]:

STOPWORDS = set(ENGLISH_STOP_WORDS)
DATA_CANDIDATES = [
    Path('/workspace/tmdb_5000_movies.csv'),
    Path('tmdb_5000_movies.csv'),
    Path('/mnt/data/tmdb_5000_movies.csv'),
]


def find_data_path():
    for path in DATA_CANDIDATES:
        if path.exists():
            return path
    raise FileNotFoundError('No se encontró tmdb_5000_movies.csv')


def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [token for token in tokens if token not in STOPWORDS and len(token) > 1]
    return tokens


## 1. Carga y limpieza de datos

In [3]:

data_path = find_data_path()
movies = pd.read_csv(data_path)
movies = movies[['title', 'overview']].copy()

print('Ruta de datos usada:', data_path)
print('Filas originales:', len(movies))

movies = movies.dropna(subset=['overview'])
movies = movies[movies['overview'].str.strip() != '']
movies = movies.drop_duplicates(subset=['title']).reset_index(drop=True)

print('Filas tras limpieza:', len(movies))
movies.head()


Ruta de datos usada: /mnt/data/tmdb_5000_movies.csv
Filas originales: 4803
Filas tras limpieza: 4796


,title,overview
0,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,Following the death of District Attorney Harve...
4,John Carter,"John Carter is a war-weary, former military ca..."


## 2. Preprocesamiento del texto

In [4]:

movies['tokens'] = movies['overview'].apply(preprocess_text)
movies = movies[movies['tokens'].apply(len) > 0].reset_index(drop=True)

print('Películas con sinopsis válidas:', len(movies))
print('Ejemplo de tokens:')
print(movies.loc[0, 'tokens'][:20])


Películas con sinopsis válidas: 4796
Ejemplo de tokens:
['nd', 'century', 'paraplegic', 'marine', 'dispatched', 'moon', 'pandora', 'unique', 'mission', 'torn', 'following', 'orders', 'protecting', 'alien', 'civilization']


## 3. Entrenamiento del modelo Word2Vec

In [5]:

embedding_model = Word2Vec(
    sentences=movies['tokens'].tolist(),
    vector_size=200,
    window=8,
    min_count=2,
    workers=1,
    sg=1,
    epochs=20,
    seed=42,
)

print('Tamaño del vocabulario:', len(embedding_model.wv))
print('Dimensión del embedding:', embedding_model.vector_size)


Tamaño del vocabulario: 11384
Dimensión del embedding: 200


## 4. Vectorización de las sinopsis

In [6]:

def overview_to_vector(tokens, embedding_model):
    vectors = [embedding_model.wv[word] for word in tokens if word in embedding_model.wv]
    if not vectors:
        return np.zeros(embedding_model.vector_size, dtype=float)
    return np.mean(vectors, axis=0)


movies['overview_vector'] = movies['tokens'].apply(lambda tokens: overview_to_vector(tokens, embedding_model))
movie_vectors = np.vstack(movies['overview_vector'].values)

print('Forma de la matriz de vectores:', movie_vectors.shape)


Forma de la matriz de vectores: (4796, 200)


## 5. Control de palabras fuera de vocabulario

In [7]:

def vocabulary_report(tokens, embedding_model):
    known_words = [word for word in tokens if word in embedding_model.wv]
    unknown_words = [word for word in tokens if word not in embedding_model.wv]
    return {
        'total_words': len(tokens),
        'known_words': len(known_words),
        'unknown_words': len(unknown_words),
    }

sample_title = 'Star Wars'
sample_tokens = movies.loc[movies['title'].str.lower() == sample_title.lower(), 'tokens'].iloc[0]
vocabulary_report(sample_tokens, embedding_model)


{'total_words': 30, 'known_words': 28, 'unknown_words': 2}

## 6. Recomendador con similitud del coseno

In [8]:

def find_title_index(movies, title):
    matches = movies.index[movies['title'].str.lower() == title.lower()].tolist()
    if matches:
        return matches[0]

    suggestions = difflib.get_close_matches(title, movies['title'].tolist(), n=5, cutoff=0.5)
    if suggestions:
        raise ValueError(f"Película no encontrada. Prueba con una de estas: {', '.join(suggestions)}")
    raise ValueError('Película no encontrada en el dataset.')


def recommend_movies(title, movies, movie_vectors, top_n=10):
    movie_index = find_title_index(movies, title)
    similarities = cosine_similarity([movie_vectors[movie_index]], movie_vectors)[0]
    sorted_indices = np.argsort(similarities)[::-1]
    sorted_indices = [index for index in sorted_indices if index != movie_index][:top_n]

    recommendations = movies.loc[sorted_indices, ['title']].copy()
    recommendations['similarity'] = similarities[sorted_indices]
    recommendations = recommendations.reset_index(drop=True)
    return recommendations


## 7. Ejemplos de recomendaciones

In [9]:

recommend_movies('Star Wars', movies, movie_vectors, top_n=10)


,title,similarity
0,The Empire Strikes Back,0.940344
1,Return of the Jedi,0.909539
2,Star Wars: Episode III - Revenge of the Sith,0.885792
3,Star Wars: Episode I - The Phantom Menace,0.861469
4,Dragon Blade,0.860653
5,Star Wars: Episode II - Attack of the Clones,0.857652
6,Conan the Destroyer,0.846879
7,Shanghai Noon,0.846622
8,Teenage Mutant Ninja Turtles: Out of the Shadows,0.835117
9,Iron Man,0.833337


In [10]:

recommend_movies('Batman', movies, movie_vectors, top_n=10)


,title,similarity
0,The Dark Knight,0.862143
1,Batman & Robin,0.845171
2,Batman Begins,0.830761
3,The Dark Knight Rises,0.825479
4,Hobo with a Shotgun,0.820649
5,Batman Returns,0.816794
6,RoboCop 3,0.811598
7,Teenage Mutant Ninja Turtles,0.808219
8,Quo Vadis,0.804675
9,The Thief and the Cobbler,0.800580


In [11]:

recommend_movies('Shrek 2', movies, movie_vectors, top_n=10)


,title,similarity
0,Shrek the Third,0.888682
1,Shrek Forever After,0.882009
2,Where the Wild Things Are,0.844570
3,Ella Enchanted,0.831617
4,Cinderella,0.829041
5,The King's Speech,0.822818
6,It Happened One Night,0.818911
7,Hardflip,0.806541
8,Monster-in-Law,0.804848
9,The Quiet,0.799329



## 8. Conclusión

El sistema devuelve recomendaciones a partir de la similitud entre sinopsis. La representación de cada película se obtiene promediando los embeddings de las palabras conocidas por el modelo Word2Vec, y la recomendación final se basa en la similitud del coseno.
